In [1]:
# ==========================================
# CELL 1: IMPORTS, CONFIGURATION & BACKUP
# ==========================================
import os
import shutil
import datetime
import numpy as np
import joblib

from collections import Counter
from imblearn.over_sampling import SMOTE

from sklearn.model_selection import train_test_split
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# ── DATA PATHS ───────────────────────────────────────────
X_PATH = r"C:\Users\kalig\OneDrive\Desktop\train_X.npy"
Y_PATH = r"C:\Users\kalig\OneDrive\Desktop\train_y.npy"

# ── MODEL PATHS ──────────────────────────────────────────
MODEL_SAVE_PATH = r"C:\Users\kalig\OneDrive\Desktop\tissue_density_model_adaboost.joblib"
MODEL_BACKUP_DIR = r"C:\Users\kalig\OneDrive\Desktop\model_backups"

RANDOM_STATE = 42
os.makedirs(MODEL_BACKUP_DIR, exist_ok=True)

print("=== ADABOOST MULTICLASS CONFIGURATION ===")
print(f"Feature file : {X_PATH}")
print(f"Label file   : {Y_PATH}")
print(f"Model output : {MODEL_SAVE_PATH}")

if os.path.exists(MODEL_SAVE_PATH):
    timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    backup_path = os.path.join(
        MODEL_BACKUP_DIR,
        f"tissue_density_model_adaboost_{timestamp}.joblib"
    )
    shutil.copy2(MODEL_SAVE_PATH, backup_path)
    print(f" Existing model backed up to: {backup_path}")
else:
    print("No existing model found — fresh training run.")

=== ADABOOST MULTICLASS CONFIGURATION ===
Feature file : C:\Users\kalig\OneDrive\Desktop\train_X.npy
Label file   : C:\Users\kalig\OneDrive\Desktop\train_y.npy
Model output : C:\Users\kalig\OneDrive\Desktop\tissue_density_model_adaboost.joblib
No existing model found — fresh training run.


In [2]:
# ==========================================
# CELL 2: LOAD, SPLIT & BALANCE DATA
# ==========================================
print("\nLoading feature matrix and labels...")
X = np.load(X_PATH)
y = np.load(Y_PATH)

print(f"Data loaded successfully! Shape: X={X.shape}, y={y.shape}")

unique, counts = np.unique(y, return_counts=True)
print(f"Full dataset class distribution: {dict(zip(unique, counts))}")

# Split first so the test set stays untouched
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print(f"\nBefore SMOTE — training distribution: {Counter(y_train)}")

smote = SMOTE(
    sampling_strategy='auto',
    random_state=RANDOM_STATE,
    k_neighbors=7
)

X_train_balanced, y_train_balanced = smote.fit_resample(X_train, y_train)

print(f"After SMOTE — training distribution: {Counter(y_train_balanced)}")

real_train_images = X_train.shape[0]
total_train_images = X_train_balanced.shape[0]
synthetic_images = total_train_images - real_train_images
test_images = X_test.shape[0]

print(f"\n{'='*52}")
print("DATA SUMMARY")
print(f"{'='*52}")
print(f"Original training images : {real_train_images}")
print(f"Synthetic images added   : {synthetic_images}")
print(f"Total training images    : {total_train_images}")
print(f"Test images              : {test_images}")
print(f"Features per image       : {X_train_balanced.shape[1]}")
print(f"{'='*52}")


Loading feature matrix and labels...
Data loaded successfully! Shape: X=(9999, 1792), y=(9999,)
Full dataset class distribution: {np.int64(1): np.int64(1172), np.int64(2): np.int64(4282), np.int64(3): np.int64(3991), np.int64(4): np.int64(554)}

Before SMOTE — training distribution: Counter({np.int64(2): 3425, np.int64(3): 3193, np.int64(1): 938, np.int64(4): 443})
After SMOTE — training distribution: Counter({np.int64(1): 3425, np.int64(2): 3425, np.int64(3): 3425, np.int64(4): 3425})

DATA SUMMARY
Original training images : 7999
Synthetic images added   : 5701
Total training images    : 13700
Test images              : 2000
Features per image       : 1792


In [3]:
# ==========================================
# CELL 3: BUILD ADABOOST MODEL
# ==========================================
print("\nInitializing multiclass AdaBoost (default sklearn behavior)...")

base_estimator = DecisionTreeClassifier(
    max_depth=1,
    random_state=RANDOM_STATE
)

model = AdaBoostClassifier(
    estimator=base_estimator,
    n_estimators=300,
    learning_rate=0.75,
    random_state=RANDOM_STATE
)

print("Training AdaBoost model on balanced data...")
model.fit(X_train_balanced, y_train_balanced)
print(" Training complete!")


Initializing multiclass AdaBoost (default sklearn behavior)...
Training AdaBoost model on balanced data...
 Training complete!


In [4]:
# ==========================================
# CELL 4: EVALUATE MODEL
# ==========================================
print("\nEvaluating AdaBoost model on test set...")

y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)

acc = accuracy_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)

print(f"\n{'='*52}")
print("PERFORMANCE REPORT")
print(f"{'='*52}")
print(f"Images model trained on : {total_train_images}")
print(f" ↳ Real images          : {real_train_images}")
print(f" ↳ SMOTE synthetic      : {synthetic_images}")
print(f"Images tested on        : {test_images}")
print(f"Overall accuracy        : {acc:.4f}")
print(f"{'='*52}")

print("\nPer-class breakdown:")
print(classification_report(
    y_test,
    y_pred,
    target_names=[
        'Tissue Density Category 1',
        'Tissue Density Category 2',
        'Tissue Density Category 3',
        'Tissue Density Category 4'
    ],
    zero_division=0
))

print("Confusion Matrix:")
header = "              " + "  ".join(f"Pred {i+1}" for i in range(4))
print(header)
for i, row in enumerate(cm):
    print(f"Actual {i+1}     " + "  ".join(f"{v:6d}" for v in row))

print("\nFirst 5 class-probability rows:")
print(y_proba[:5])


Evaluating AdaBoost model on test set...

PERFORMANCE REPORT
Images model trained on : 13700
 ↳ Real images          : 7999
 ↳ SMOTE synthetic      : 5701
Images tested on        : 2000
Overall accuracy        : 0.5965

Per-class breakdown:
                           precision    recall  f1-score   support

Tissue Density Category 1       0.44      0.82      0.57       234
Tissue Density Category 2       0.68      0.53      0.60       857
Tissue Density Category 3       0.71      0.58      0.64       798
Tissue Density Category 4       0.33      0.74      0.46       111

                 accuracy                           0.60      2000
                macro avg       0.54      0.67      0.57      2000
             weighted avg       0.65      0.60      0.60      2000

Confusion Matrix:
              Pred 1  Pred 2  Pred 3  Pred 4
Actual 1        193      39       2       0
Actual 2        240     452     156       9
Actual 3          8     169     466     155
Actual 4          0     

In [5]:
# ==========================================
# CELL 5: CLASS CONFIDENCE SNAPSHOT
# ==========================================
print("\nSample class-probability breakdown (first 10 test images):\n")

header = (
    f"{'Image':<8} | "
    f"{'Cat 1':>6} {'Cat 2':>6} {'Cat 3':>6} {'Cat 4':>6} | "
    f"{'Pred':>5} {'True':>5}"
)
print(header)
print("-" * len(header))

for i in range(min(10, len(X_test))):
    p_row = y_proba[i]
    pred = y_pred[i]
    true = y_test[i]
    match = "✅" if pred == true else "❌"

    print(
        f"{i+1:<8} | "
        f"{p_row[0]:>6.3f} {p_row[1]:>6.3f} {p_row[2]:>6.3f} {p_row[3]:>6.3f} | "
        f"{pred:>5} {true:>5}  {match}"
    )


Sample class-probability breakdown (first 10 test images):

Image    |  Cat 1  Cat 2  Cat 3  Cat 4 |  Pred  True
----------------------------------------------------
1        |  0.235  0.254  0.257  0.255 |     3     3  ✅
2        |  0.232  0.252  0.259  0.257 |     3     3  ✅
3        |  0.232  0.253  0.257  0.259 |     4     3  ❌
4        |  0.247  0.256  0.258  0.238 |     3     2  ❌
5        |  0.224  0.252  0.260  0.264 |     4     3  ❌
6        |  0.263  0.258  0.252  0.228 |     1     2  ❌
7        |  0.251  0.258  0.256  0.235 |     2     2  ✅
8        |  0.244  0.254  0.255  0.247 |     3     3  ✅
9        |  0.247  0.256  0.254  0.242 |     2     2  ✅
10       |  0.227  0.255  0.260  0.258 |     3     4  ❌


In [6]:
# ==========================================
# CELL 6: SAVE MODEL
# ==========================================
print(f"\nSaving model to {MODEL_SAVE_PATH}...")
joblib.dump(model, MODEL_SAVE_PATH)
print(" Done! AdaBoost tissue density model saved.")


Saving model to C:\Users\kalig\OneDrive\Desktop\tissue_density_model_adaboost.joblib...
 Done! AdaBoost tissue density model saved.


In [7]:
# ==========================================
# CELL 7: PREDICTION UTILITY
# ==========================================
def predict_new(X_new: np.ndarray):
    """
    Returns:
      preds       -> predicted tissue density class labels
      class_probs -> multiclass AdaBoost probabilities
    """
    preds = model.predict(X_new)
    class_probs = model.predict_proba(X_new)
    return preds, class_probs

print(" Prediction utility ready.")
print("Use: preds, class_probs = predict_new(X_new)")

 Prediction utility ready.
Use: preds, class_probs = predict_new(X_new)
